# 06 -- Backend

FastAPI application startup, database/persistence layer, security hardening, CI/CD, containerization, deployment, and real production incidents -- illustrative excerpts of the actual backend source, extracted from the appendix of the original notebook.

## 9. Backend API (FastAPI)

`backend/app/main.py` wires everything together: model loading at startup (once, cached
in `app.state.model_registry`), migrations, CORS, rate limiting, request timing.


In [ ]:
# backend/app/main.py -- application startup (real code)
@asynccontextmanager
async def lifespan(app: FastAPI):
    registry = ModelRegistry()
    registry.load_all()                 # BERT/CNN2D/RFM loaded ONCE, not per-request
    app.state.model_registry = registry

    repo = AnalyticsRepository()
    repo.load_all()
    app.state.analytics_repository = repo

    _check_metrics_freshness()          # warns if a checkpoint was retrained without
                                         # regenerating the metrics that describe it
    if db_configured():
        _run_pending_migrations()       # Alembic runs here -- see Section 16
    yield

app.add_middleware(CORSMiddleware, allow_origins=settings.FRONTEND_ORIGINS,
                    allow_methods=["GET", "POST"], allow_headers=["Content-Type", "X-API-Key"])


**Endpoints** (`app/api/v1/`): `/sentiment/predict`, `/predict-batch`, `/pipeline`
(all 3 tasks together), `/explain` (SHAP), `/upload-file` (batch CSV/Excel), `/analyses`
(history), plus read-only analytics endpoints backing the dashboard (`/analytics/*`,
`/segmentation/*`, `/customers/*`, `/products/*`).

**Idempotency**: `POST /predict` accepts an `Idempotency-Key` header — if a client retries
after a network timeout, the server replays the already-saved result instead of creating a
duplicate history row (`SentimentAnalysis.idempotency_key`, unique-constrained).

**Concurrency & timeout control on uploads**: bounded chunked reads (5MB cap), a semaphore
capping concurrent batch-classification work at 2, and a 300s timeout — all added after a
security/reliability audit found the unbounded version was a real memory-exhaustion vector
on a 512MB host.


## 10. Database & persistence layer

**Why it exists**: this is a static-dataset analytics app (no user accounts, no CRUD on
Olist's own data) — the one real persistence gap was that AI predictions and batch-upload
results were never saved, or saved as local JSON files that don't survive a redeploy on
Render's ephemeral disk.


In [ ]:
# backend/app/db/models.py -- the 4 tables, and why exactly these 4 (real docstring)
'''
Scope note: this project has no authentication, no user accounts, and no
"create/edit review" journey anywhere -- the Olist orders/customers/reviews
data is a static analytics dataset (parquet/JSON), not something users CRUD.
So there are deliberately no Users/Orders/Products/Auth tables here.
'''
# SentimentAnalysis        -- one row per /predict call, unique idempotency_key
# SentimentAnalysisAspect  -- normalized (queried per-aspect independently)
# PredictionFeedback       -- thumbs up/down on a prediction
# BatchUploadJob           -- full batch result stored as ONE JSON blob per job,
#                             not normalized row-by-row (no query pattern needs that)


**Verified, not assumed, in sync**: `alembic check` run against a fresh database with
every migration applied reports *"No new upgrade operations detected"* — the migrations
and the current `models.py` are provably consistent, checked as part of this walkthrough.

**A real production bug this caught**: `DATABASE_URL` being set and connectable made
`/health` report `"connected": true` even when the schema had zero tables (a fresh
database that had never run its migrations) — every write then silently failed inside
the best-effort persistence layer. Fixed by running Alembic migrations automatically at
app startup (`_run_pending_migrations()` in Section 9) instead of relying on a separate
manual release step that Render's simple deploy flow doesn't have.


## 12. Security & reliability hardening

A structured technical review (22 numbered issues, 5 phases) covered results integrity,
security/availability, ML correctness, engineering discipline, and test coverage. A sample
of the concrete fixes, each with a real failure mode it closes:


In [ ]:
# Phase 2 -- Security/availability
# 1. CORS: allow_methods=["*"] / allow_headers=["*"]  ->  explicit ["GET","POST"] /
#    ["Content-Type","X-API-Key"] -- this API never needs anything broader.
# 2. Rate limiting keyed on the RAW socket peer -- behind Render's reverse proxy, that's
#    always the proxy's own address, so every real client shared ONE limit bucket.
#    Fixed: parse X-Forwarded-For at a configured TRUSTED_PROXY_HOPS position.
# 3. Path traversal: upload_id read directly into a filesystem path with no validation.
#    Fixed: strict 32-hex regex + resolved-path containment check before any file I/O.

# Phase 3 -- ML correctness
# 4. RFM train/serve skew (Section 5) -- pipeline object reused, not reimplemented.
# 5. ABSA hallucination (Section 7) -- keyword-presence gate added.

# Phase 4 -- Engineering discipline
# 6. Dockerfile ran as root, no HEALTHCHECK, no --proxy-headers (rate limiter saw the
#    proxy's IP for every request without it) -- see Section 14.
# 7. requirements.txt had no upper bounds -- a fresh install could silently resolve a
#    breaking dependency combination months later with zero warning (this happened live,
#    see Section 16).


**Verification discipline used throughout**: every fix in this project was checked
against real data/tests, not just read for plausibility -- 129+ backend tests, real
Alembic migrations run against a live database, real curl calls against the deployed API
after each change.


## 13. CI/CD pipeline (GitHub Actions)

Three jobs, on every push to `main` and every PR:


```yaml
# .github/workflows/ci.yml (real)
jobs:
  backend:
    steps:
      - uses: actions/checkout@v4
        with: { lfs: true }              # model checkpoints are git-lfs tracked
      - run: pip install -r backend/requirements-dev.txt
      - run: cd backend && pytest -q
      - run: cd backend && python scripts/check_no_local_paths.py     # catches leaked
                                                                        # machine-specific paths
      - run: cd backend && python scripts/verify_metrics_freshness.py # catches a stale
                                                                        # metrics/checkpoint mismatch
  frontend:
    steps:
      - run: cd frontend && npm ci && npm run typecheck && npm test && npm run build
  docker:
    steps:
      - run: docker build -f backend/Dockerfile -t baseera-api .
      - run: |                            # boots the REAL image and health-checks it --
          docker run --rm -d -p 8000:8000 baseera-api    # not just "does it build"
          for i in $(seq 1 20); do curl -fsS http://localhost:8000/api/v1/health && exit 0; sleep 5; done

```


`check_no_local_paths.py` and `verify_metrics_freshness.py` both exist because of real
bugs this project found — a Windows user-profile path once leaked into a committed results
file, and a checkpoint was overwritten by a retraining run without regenerating the metrics
file describing it.


## 14. Containerization (Docker, multi-stage)


```dockerfile
# backend/Dockerfile (real, final version)
FROM python:3.11-slim AS builder
WORKDIR /build
COPY backend/requirements.txt .
RUN pip install --no-cache-dir --prefix=/install \
    --extra-index-url https://download.pytorch.org/whl/cpu \      # CPU-only torch wheel:
    -r requirements.txt                                             # this app never touches
                                                                      # a GPU in production
                                                                      # (~200MB vs ~2GB+ CUDA)
FROM python:3.11-slim
RUN useradd --create-home --uid 10001 appuser        # non-root
COPY --from=builder /install /usr/local               # only installed packages, not build tools
COPY --chown=appuser:appuser backend /app/backend
COPY --chown=appuser:appuser models /app/models
USER appuser
HEALTHCHECK --interval=30s --timeout=5s --start-period=90s --retries=3 \
    CMD python -c "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://localhost:8000/api/v1/health').status==200 else 1)"
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", \
     "--proxy-headers", "--forwarded-allow-ips", "*"]   # without this, rate limiting sees
                                                          # the proxy's IP for every request

```


Verified via a dedicated `harden-dockerfile` branch + PR, gated on the CI `docker` job
(genuine build + boot + `/health` check) passing *before* merging to `main` — not shipped
straight to the live deployment unverified.


## 15. Deployment (Render + Vercel)

**Backend** (Render, free web-service tier, Docker runtime): auto-deploys on every push to
`main`. `ENABLE_BERT=false` there (512MB RAM can't fit BERT's 670MB alongside PyTorch's own
overhead) — the public link serves CNN2D.

**Frontend** (Vercel, static build): `VITE_API_BASE_URL` points at the Render backend;
`FRONTEND_ORIGINS` on the backend must list the exact Vercel URL for CORS to allow it.


## 16. Production incidents, found and fixed live

Real bugs caught by actually checking the live deployment (not just assuming a push =
a successful deploy), each with root cause and fix:

### 16.1 — Silent OOM: every deploy since a specific commit had been failing

A freshness check (`_check_metrics_freshness()`, Section 9) hashed the full ~670MB BERT
checkpoint into memory via `Path.read_bytes()` on **every startup**, regardless of
`ENABLE_BERT`. On Render's 512MB instance this alone exceeded the memory limit and killed
the deploy — silently, because the *previous* successful deploy kept serving `/health` as
"healthy" the whole time.


In [ ]:
# The bug (backend/app/ml/utils.py, before the fix)
def checkpoint_fingerprint(model_dir):
    h = hashlib.sha256()
    for p in weight_files:
        h.update(p.read_bytes())   # loads the ENTIRE file into RAM at once
    return h.hexdigest()[:16]

# The fix -- stream in 1MB chunks (identical hash output, verified against the
# already-recorded checkpoint_sha256), and skip the check entirely when the model
# it would fingerprint is never even loaded (ENABLE_BERT=false)
def checkpoint_fingerprint(model_dir):
    h = hashlib.sha256()
    for p in weight_files:
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
    return h.hexdigest()[:16]


### 16.2 — CORS misconfiguration silently broke the entire public dashboard

`FRONTEND_ORIGINS` on Render didn't match the actual deployed Vercel URL — every dashboard
API call failed with `NETWORK_ERROR` in the browser console, despite the backend itself
being 100% healthy. Caught by opening the live site and reading real browser console
errors, not by assuming a passing backend health check meant the whole system worked.

### 16.3 — sklearn cross-version unpickling risk

`InconsistentVersionWarning` surfaced in the live logs: the RFM scaler/K-Means pickles had
been saved with a locally-installed scikit-learn newer than the version
`requirements.txt` actually pins for production. Flagged for re-pickling with the
production-matching version — the exact failure mode `requirements.txt`'s own upper bound
comment already warned about.


## 17. Final verification & conclusion

Everything in this notebook is checkable against the live system right now:


### What this project demonstrates, end to end

- **A real, quantified data-quality bug** (train/test leakage) found in the starting
  notebook and fixed with a verifiable, zero-overlap split.
- **Two independently trained sentiment models**, evaluated on a leak-free test set, with
  their decision threshold and calibration empirically checked rather than assumed.
- **A fake-review detector rebuilt from scratch** after two prior checkpoints were
  confirmed unreliable — including rejecting a plausible-looking dataset *before* wasting
  training time on it, and validating the final model with a statistically meaningful
  sample (320 reviews, Wilson confidence intervals), not a handful of cherry-picked examples.
- **A full production system**: FastAPI backend, typed React frontend, a persistence layer
  with verified migration/model sync, CI that actually boots and health-checks a real
  Docker container, and — critically — real production incidents that were found by
  checking the live deployment directly and fixed with a measured root cause, not guessed at.

Every number in this notebook has a `results/*.json` file behind it, every code excerpt
points at a real file in this repository, and every "fixed" claim was re-verified against
either the test suite or the live deployment before being written down here.
